# Setup Verification

Run all cells top-to-bottom. The final cell prints a pass/fail summary.

**Before running:** activate the env and launch jupyter from the repo root:
```bash
conda activate qai
cd /Applications/Projects/Hackathons/quantum_ai
jupyter notebook setup_check.ipynb
```

Each cell appends to a global `RESULTS` dict; the last cell summarises. Cells fail soft — one broken check won't stop the rest.

In [1]:
import sys, os, importlib, traceback

RESULTS = {}

def check(name, fn):
    try:
        out = fn()
        msg = out if isinstance(out, str) else 'ok'
        RESULTS[name] = ('PASS', msg)
        print(f'[PASS] {name}: {msg}')
    except Exception as e:
        msg = f'{type(e).__name__}: {str(e).splitlines()[0]}'
        RESULTS[name] = ('FAIL', msg)
        print(f'[FAIL] {name}: {msg}')

check('python_version',    lambda: f'{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}')
check('conda_env',          lambda: os.environ.get('CONDA_DEFAULT_ENV') or '(not in conda)')
check('working_dir',        lambda: os.getcwd())

[PASS] python_version: 3.11.15
[PASS] conda_env: qai
[PASS] working_dir: /Applications/Projects/Hackathons/quantum_ai


## Fix missing packages + broken pydantic_core

Run this once. It installs missing packages into the kernel's Python and forces a clean reinstall of pydantic (which fixes the x86_64/arm64 mismatch from a stale user-site install). **Restart the kernel after this cell finishes.**

In [2]:
import sys, subprocess, os, shutil, glob

# 1. Nuke the broken x86_64 pydantic_core in user-site (it shadows the conda-env one)
user_site = os.path.expanduser('~/.local/lib/python3.11/site-packages')
for pattern in ['pydantic_core*', 'pydantic-*', 'pydantic']:
    for path in glob.glob(os.path.join(user_site, pattern)):
        print(f'  removing {path}')
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
        else:
            try: os.remove(path)
            except: pass

# 2. Install missing packages into kernel Python
print('\nInstalling missing packages into', sys.executable)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm', 'faiss-cpu', 'chronos-forecasting', 'autogluon.timeseries'])

# 3. Reinstall pydantic + pydantic_core TOGETHER so pip resolves matching versions.
#    Previous version did `--no-deps pydantic-core` then `--no-deps pydantic`, which
#    pulled the *latest* pydantic-core (2.47.0) while pydantic 2.13.4 pins 2.46.4 —
#    that's exactly what was causing the SystemError on every run.
print('\nReinstalling pydantic stack with matching versions...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', 'pydantic==2.13.4', 'pydantic-core==2.46.4'])

print('\nDone. RESTART KERNEL now (Kernel -> Restart), then re-run all cells.')


Installing missing packages into /opt/anaconda3/envs/qai/bin/python

Reinstalling pydantic stack with matching versions...

Done. RESTART KERNEL now (Kernel -> Restart), then re-run all cells.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.


## Quantum stack

In [3]:
for mod in ['qiskit', 'qiskit_aer', 'qiskit_algorithms', 'qiskit_optimization', 'qiskit_ibm_runtime', 'pennylane']:
    check(f'import {mod}', lambda m=mod: importlib.import_module(m).__name__)

[PASS] import qiskit: qiskit
[PASS] import qiskit_aer: qiskit_aer
[PASS] import qiskit_algorithms: qiskit_algorithms
[PASS] import qiskit_optimization: qiskit_optimization
[PASS] import qiskit_ibm_runtime: qiskit_ibm_runtime
[PASS] import pennylane: pennylane


## ML stack

In [4]:
for mod in ['torch', 'transformers', 'huggingface_hub', 'datasets', 'peft', 'accelerate']:
    check(f'import {mod}', lambda m=mod: importlib.import_module(m).__name__)

[PASS] import torch: torch


/Users/dhyutimac/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] import transformers: transformers
[PASS] import huggingface_hub: huggingface_hub
[PASS] import datasets: datasets
[PASS] import peft: peft
[PASS] import accelerate: accelerate


## Finance stack

In [5]:
for mod in ['yfinance', 'alpaca', 'polygon', 'cvxpy', 'pypfopt', 'backtrader', 'lightgbm']:
    check(f'import {mod}', lambda m=mod: importlib.import_module(m).__name__)

[PASS] import yfinance: yfinance
[PASS] import alpaca: alpaca
[PASS] import polygon: polygon
[PASS] import cvxpy: cvxpy
[PASS] import pypfopt: pypfopt
[PASS] import backtrader: backtrader
[PASS] import lightgbm: lightgbm


## LLM / agent stack

In [6]:
for mod in ['langchain', 'chromadb', 'faiss', 'pydantic']:
    check(f'import {mod}', lambda m=mod: importlib.import_module(m).__name__)

[PASS] import langchain: langchain
[PASS] import chromadb: chromadb
[PASS] import faiss: faiss
[PASS] import pydantic: pydantic


## Foundation models

In [7]:
for mod in ['chronos', 'autogluon']:
    check(f'import {mod}', lambda m=mod: importlib.import_module(m).__name__)

[PASS] import chronos: chronos
[PASS] import autogluon: autogluon


## Environment variables

If any of these say `NOT SET`, add them to your `~/.zshrc` and restart the kernel:
```bash
export ALPACA_API_KEY=...
export ALPACA_SECRET_KEY=...
export POLYGON_API_KEY=...
export XPYQ_API_KEY=...   # optional, used in the XpyQ SDK check below
```

In [8]:
for k in ['ALPACA_API_KEY', 'ALPACA_SECRET_KEY', 'POLYGON_API_KEY', 'XPYQ_API_KEY']:
    val = os.environ.get(k)
    if val:
        masked = (val[:4] + '...' + val[-4:]) if len(val) > 8 else '***'
        RESULTS[f'env {k}'] = ('PASS', masked)
        print(f'[PASS] env {k}: {masked}')
    else:
        RESULTS[f'env {k}'] = ('FAIL', 'NOT SET')
        print(f'[FAIL] env {k}: NOT SET')

[PASS] env ALPACA_API_KEY: PKUI...GIIE
[PASS] env ALPACA_SECRET_KEY: BorH...TrPr
[PASS] env POLYGON_API_KEY: IOLa...CEjD
[PASS] env XPYQ_API_KEY: xpyq...5f80


## IBM Quantum saved account

If this fails, save your token once:
```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_KEY', overwrite=True)
```

In [17]:
  from qiskit_ibm_runtime import QiskitRuntimeService
  QiskitRuntimeService.save_account(
      channel='ibm_quantum_platform',
      token='crn:v1:bluemix:public:quantum-computing:us-east:a/80bf48f2b810459381fd301b2234feb8:0ba1ebdd-dd7d-47af-8d0c-6f2b502a4ef8::',
      overwrite=True,
  )

In [18]:
def ibm_check():
    from qiskit_ibm_runtime import QiskitRuntimeService
    accts = QiskitRuntimeService.saved_accounts()
    if not accts:
        raise RuntimeError('no saved IBM Quantum accounts')
    return f'{len(accts)} account(s) saved: {list(accts.keys())}'

check('IBM Quantum saved account', ibm_check)

[PASS] IBM Quantum saved account: 1 account(s) saved: ['default-ibm-quantum-platform']


## Live API smoke tests

In [10]:
def yf_check():
    import yfinance as yf
    df = yf.download('AAPL', period='5d', progress=False, auto_adjust=True)
    if df.empty:
        raise RuntimeError('empty result')
    # yfinance returns multi-index columns even for single ticker in newer versions
    close = df['Close']
    last = close.iloc[-1]
    if hasattr(last, 'iloc'):
        last = last.iloc[0]
    return f'got {len(df)} rows, last close ${float(last):.2f}'

check('yfinance live (AAPL 5d)', yf_check)

[PASS] yfinance live (AAPL 5d): got 5 rows, last close $312.06


In [11]:
def polygon_check():
    import requests
    k = os.environ.get('POLYGON_API_KEY')
    if not k:
        raise RuntimeError('POLYGON_API_KEY not set')
    r = requests.get(f'https://api.polygon.io/v2/aggs/ticker/AAPL/prev?apiKey={k}', timeout=10)
    r.raise_for_status()
    d = r.json()
    if 'results' not in d or not d['results']:
        raise RuntimeError(f"unexpected response: {str(d)[:120]}")
    return f"AAPL prev close ${d['results'][0]['c']}"

check('Polygon live (AAPL prev close)', polygon_check)

[PASS] Polygon live (AAPL prev close): AAPL prev close $312.06


In [12]:
def alpaca_check():
    import requests
    k, s = os.environ.get('ALPACA_API_KEY'), os.environ.get('ALPACA_SECRET_KEY')
    if not (k and s):
        raise RuntimeError('ALPACA_API_KEY / ALPACA_SECRET_KEY not set')
    r = requests.get(
        'https://paper-api.alpaca.markets/v2/account',
        headers={'APCA-API-KEY-ID': k, 'APCA-API-SECRET-KEY': s},
        timeout=10,
    )
    r.raise_for_status()
    a = r.json()
    return f"paper account status={a.get('status')}, cash=${a.get('cash')}"

check('Alpaca live (paper account)', alpaca_check)

[PASS] Alpaca live (paper account): paper account status=ACTIVE, cash=$100000


## Quantum circuit smoke tests

In [13]:
def qiskit_bell():
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1); qc.measure([0, 1], [0, 1])
    counts = AerSimulator().run(qc, shots=1024).result().get_counts()
    keys = set(counts.keys())
    if not ({'00', '11'} & keys):
        raise RuntimeError(f'unexpected counts: {counts}')
    return f'counts={counts}'

check('Qiskit Bell state (AerSim)', qiskit_bell)

[PASS] Qiskit Bell state (AerSim): counts={'00': 512, '11': 512}


In [14]:
def pennylane_check():
    import pennylane as qml
    from pennylane import numpy as pnp
    dev = qml.device('default.qubit', wires=2)

    @qml.qnode(dev)
    def circuit(x):
        qml.RX(x, wires=0)
        qml.CNOT(wires=[0, 1])
        return qml.expval(qml.PauliZ(1))

    val = circuit(pnp.array(0.5))
    return f'<Z>={float(val):.4f}'

check('PennyLane QNode (default.qubit)', pennylane_check)

[PASS] PennyLane QNode (default.qubit): <Z>=0.8776


## XpyQ — REST API

Set `XPYQ_KEY` env var (or `XPYQ_API_KEY`) to your `xpyq_live_...` key from xpyq.ai dashboard.

```bash
echo 'export XPYQ_KEY=xpyq_live_90a721fc099d665e451ada539f4d50a1' >> ~/.zshrc
```

This check pings `/api/v1/compute/account` for credits, then submits a 2x2 matmul to verify hardware access.

In [15]:
def xpyq_check():
    import requests, time

    BASE = 'https://xpyq-lib-production.up.railway.app'
    # Accept either env var name
    k = os.environ.get('XPYQ_KEY') or os.environ.get('XPYQ_API_KEY')
    if not k:
        raise RuntimeError('XPYQ_KEY (or XPYQ_API_KEY) env var not set')

    H = {'Authorization': f'Bearer {k}', 'Content-Type': 'application/json'}

    # 1. Account endpoint — cheap auth ping
    r = requests.get(f'{BASE}/api/v1/compute/account', headers=H, timeout=15)
    if r.status_code == 401:
        raise RuntimeError(f'XPYQ key rejected (401)')
    r.raise_for_status()
    acct = r.json()
    credits = acct.get('credits', '?')

    # 2. Submit a small matmul, poll for completion
    code = """import numpy as _np
A = from_numpy(_np.array([[1, 2], [3, 4]], dtype=_np.float32))
B = from_numpy(_np.array([[5, 6], [7, 8]], dtype=_np.float32))
result = matmul(A, B).numpy()
print(result.tolist())"""
    submit = requests.post(
        f'{BASE}/api/v1/compute/runs',
        headers=H,
        json={'code': code, 'name': 'healthcheck'},
        timeout=15,
    )
    submit.raise_for_status()
    run_id = submit.json()['run_id']

    # 3. Poll until terminal (queue can be slow; allow up to 120s).
    deadline = time.time() + 120
    poll = None
    last_status = None
    while time.time() < deadline:
        poll = requests.get(f'{BASE}/api/v1/compute/runs/{run_id}', headers=H, timeout=10).json()
        last_status = poll.get('status')
        if last_status in ('completed', 'failed', 'timed_out', 'cancelled'):
            break
        time.sleep(1.0)
    else:
        raise RuntimeError(f'run {run_id} stuck in status={last_status!r} after 120s (xpyq queue slow?)')

    if poll['status'] != 'completed':
        raise RuntimeError(f'run {poll["status"]}: stdout={poll.get("stdout", "")[:200]}')

    return (f'credits={credits}, matmul={poll.get("stdout", "").strip()}, '
            f'duration_ms={poll.get("duration_ms")}, charged={poll.get("credits_charged")}cr')

check('XpyQ REST API live', xpyq_check)

[PASS] XpyQ REST API live: credits=4942, matmul=[[19.0, 22.0], [43.0, 50.0]], duration_ms=338, charged=1cr


## Summary

In [19]:
print('=' * 60)
print('SETUP CHECK SUMMARY')
print('=' * 60)

passed = [k for k, v in RESULTS.items() if v[0] == 'PASS']
failed = [k for k, v in RESULTS.items() if v[0] == 'FAIL']

print(f'\n{len(passed)} passed, {len(failed)} failed\n')

if failed:
    print('FAILED CHECKS:')
    for k in failed:
        print(f'  - {k}:  {RESULTS[k][1]}')
    print('\nFix these, then re-run the relevant cells.')
else:
    print('All checks passed. You are good to go — clone the track starter and start hacking.')

SETUP CHECK SUMMARY

39 passed, 0 failed

All checks passed. You are good to go — clone the track starter and start hacking.
